<a href="https://colab.research.google.com/github/amathie5/PPS-Project-/blob/main/pps_project_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Parameters
horizon = 12    #in months
w_i = 90        # Initial number of workers
p = 10          #Production rate per worker per month
inv_i = 900     # Initial inventory
inv_t = 1000    # Final inventory target by end of December
hold = 1000     # Inventory holding cost per watch per month (security/insurance)
wage_r = 7000   # Regular wage/worker/month
hiring = 50000  # Hiring cost per worker
layoff = 25000  # layoff cost per worker
max_overtime = 0.2  # overtime allowance......
overtime_cost = 14000 # 2x wage_r
subcontracting_cost = 15000 # CHF per watch
max_contracting = 300 # watches per month
overtime_months = {3,5,9,12} # march, may, sept, dec
subcontracting_months = {6,7,10,12} # june, july, oct, dec
demand = [900,950,1200,1050,1100,1300,1250,1100,1300,1450,1500,1700]

In [ ]:
pip install pulp numpy pandas matplotlib


In [ ]:
import numpy as np
import pandas as pd

def simulate_plan(strategy):

    months = horizon
    workers = np.zeros(months)
    hired = np.zeros(months)
    fired = np.zeros(months)
    production = np.zeros(months)
    inventory = np.zeros(months+1)

    inventory[0] = inv_i

    # ----------------------------
    # LEVEL STRATEGY
    # ----------------------------

    if strategy == "level":

        production_per_month = w_i * p
        workers[:] = w_i

        for t in range(months):

            production[t] = production_per_month

            inventory[t+1] = inventory[t] + production[t] - demand[t]

    # ----------------------------
    # CHASE STRATEGY
    # ----------------------------

    if strategy == "chase":

        for t in range(months):

            workers_needed = np.ceil(demand[t] / p)

            workers[t] = workers_needed

            if t == 0:
                hired[t] = max(0, workers[t] - w_i)
                fired[t] = max(0, w_i - workers[t])
            else:
                hired[t] = max(0, workers[t] - workers[t-1])
                fired[t] = max(0, workers[t-1] - workers[t])

            production[t] = workers[t] * p

            inventory[t+1] = inventory[t] + production[t] - demand[t]

    # ----------------------------
    # COSTS
    # ----------------------------

    wage_cost = workers * wage_r
    hiring_cost = hired * hiring
    firing_cost = fired * layoff
    inventory_cost = inventory[1:] * hold

    total_cost = wage_cost + hiring_cost + firing_cost + inventory_cost

    results = pd.DataFrame({
        "Month": range(1,months+1),
        "Workers": workers,
        "Hired": hired,
        "Fired": fired,
        "Production": production,
        "Demand": demand,
        "Inventory": inventory[1:],
        "Total Cost": total_cost
    })

    summary = {
        "Total Wage": wage_cost.sum(),
        "Total Hiring": hiring_cost.sum(),
        "Total Layoff": firing_cost.sum(),
        "Total Inventory": inventory_cost.sum(),
        "Grand Total": total_cost.sum()
    }

    return results, summary

In [ ]:
#level strategy
level_table, level_cost = simulate_plan("level")

print(level_table)
print(level_cost)

    Month  Workers  Hired  Fired  Production  Demand  Inventory  Total Cost
0       1     90.0    0.0    0.0       900.0     900      900.0   1530000.0
1       2     90.0    0.0    0.0       900.0     950      850.0   1480000.0
2       3     90.0    0.0    0.0       900.0    1200      550.0   1180000.0
3       4     90.0    0.0    0.0       900.0    1050      400.0   1030000.0
4       5     90.0    0.0    0.0       900.0    1100      200.0    830000.0
5       6     90.0    0.0    0.0       900.0    1300     -200.0    430000.0
6       7     90.0    0.0    0.0       900.0    1250     -550.0     80000.0
7       8     90.0    0.0    0.0       900.0    1100     -750.0   -120000.0
8       9     90.0    0.0    0.0       900.0    1300    -1150.0   -520000.0
9      10     90.0    0.0    0.0       900.0    1450    -1700.0  -1070000.0
10     11     90.0    0.0    0.0       900.0    1500    -2300.0  -1670000.0
11     12     90.0    0.0    0.0       900.0    1700    -3100.0  -2470000.0
{'Total Wage

In [ ]:
chase_table, chase_cost = simulate_plan("chase")

print(chase_table)
print(chase_cost)

    Month  Workers  Hired  Fired  Production  Demand  Inventory  Total Cost
0       1     90.0    0.0    0.0       900.0     900      900.0   1530000.0
1       2     95.0    5.0    0.0       950.0     950      900.0   1815000.0
2       3    120.0   25.0    0.0      1200.0    1200      900.0   2990000.0
3       4    105.0    0.0   15.0      1050.0    1050      900.0   2010000.0
4       5    110.0    5.0    0.0      1100.0    1100      900.0   1920000.0
5       6    130.0   20.0    0.0      1300.0    1300      900.0   2810000.0
6       7    125.0    0.0    5.0      1250.0    1250      900.0   1900000.0
7       8    110.0    0.0   15.0      1100.0    1100      900.0   2045000.0
8       9    130.0   20.0    0.0      1300.0    1300      900.0   2810000.0
9      10    145.0   15.0    0.0      1450.0    1450      900.0   2665000.0
10     11    150.0    5.0    0.0      1500.0    1500      900.0   2200000.0
11     12    170.0   20.0    0.0      1700.0    1700      900.0   3090000.0
{'Total Wage